# Week 8: NLP Foundations & Text Embeddings
## Notebook 09: TextRank: Graph-Based Keyword & Summary Extraction

### 1. Conceptual Intuition & Theoretical Background
**TextRank (Mihalcea & Tarau, 2004)** adapts Google's PageRank algorithm for unsupervised text summarization and keyword extraction.

**1. Keyword Extraction Graph:**
- **Nodes:** Words (filtered for nouns and adjectives).
- **Edges:** Co-occurrence within a sliding window of size $W$ (e.g. $W=3$).

**2. Sentence Summarization Graph:**
- **Nodes:** Entire sentences.
- **Edge Weights:** Normalized semantic overlap:
$$\text{Similarity}(S_i, S_j) = \frac{|\{w \mid w \in S_i \land w \in S_j\}|}{\log(|S_i|) + \log(|S_j|)}$$

**TextRank Scoring Equation:**
$$WS(V_i) = (1 - d) + d \sum_{V_j \in \text{In}(V_i)} \frac{w_{ji}}{\sum_{V_k \in \text{Out}(V_j)} w_{jk}} WS(V_j)$$
Where $d = 0.85$ is the PageRank damping factor.

---

### 2. Scratch Python Implementation: TextRank Extractor

In [ ]:
import numpy as np

def textrank_sentence_extractor(sentences, damping=0.85, max_iter=50, tol=1e-5):
    """Unsupervised TextRank sentence extractive summarization from scratch."""
    N = len(sentences)
    # Tokenize sentences into word sets
    word_sets = [set(s.lower().replace('.', '').split()) for s in sentences]
    
    # 1. Build Similarity Matrix W
    W = np.zeros((N, N))
    for i in range(N):
        for j in range(i + 1, N):
            intersection = len(word_sets[i].intersection(word_sets[j]))
            len_i = len(word_sets[i])
            len_j = len(word_sets[j])
            
            if intersection > 0 and len_i > 1 and len_j > 1:
                # Normalized logarithmic overlap
                sim = intersection / (np.log(len_i) + np.log(len_j))
                W[i, j] = sim
                W[j, i] = sim
                
    # 2. Power Iteration: PageRank stationary distribution
    # Out-degree row normalization
    row_sums = np.sum(W, axis=1, keepdims=True)
    row_sums[row_sums == 0] = 1.0 # Avoid division by zero
    M = W / row_sums
    
    # Initial scores
    scores = np.ones(N) / N
    for _ in range(max_iter):
        new_scores = (1 - damping) / N + damping * np.dot(M.T, scores)
        if np.linalg.norm(new_scores - scores) < tol:
            break
        scores = new_scores
        
    return scores

# Test Paragraph on Machine Learning Operations
document = [
    "Machine learning models are deployed into production environments to serve predictions.",
    "Data drift occurs when statistical properties of input data shift over time.",
    "Monitoring data drift is essential to maintain model accuracy and reliability.",
    "FastAPI and Docker provide low-latency microservice architectures for machine learning serving.",
    "When severe concept drift is detected, automated retraining pipelines are triggered."
]

rank_scores = textrank_sentence_extractor(document)
top_idx = np.argsort(-rank_scores)

print("--- TextRank Extractive Summarization ---")
print(f"Top 1 Representative Sentence (Score: {rank_scores[top_idx[0]]:.4f}):")
print(f"  -> '{document[top_idx[0]]}'")
print(f"\nTop 2 Representative Sentence (Score: {rank_scores[top_idx[1]]:.4f}):")
print(f"  -> '{document[top_idx[1]]}'")

### 3. Step-by-Step Code Walkthrough
1. **Graph Construction:** Measures lexical overlap between sentences.
2. **Transition Probability Matrix ($M$):** Normalizes edge weights to preserve conservation of probability mass.
3. **Power Iteration:** Converges to the principal eigenvector representing sentence centrality.

---

### 4. Industry Applications & Trade-offs
- **Legal Document Synopsis:** Fast, unsupervised executive summary extraction from lengthy contracts.
- **Trade-off:** Pure lexical overlap ignores synonyms and paraphrase variations; modern variants use dense embeddings.

---

### 5. Top Interview Questions & Answers
**Q1: How does TextRank differ from standard PageRank?**
> **Answer:** PageRank runs on directed unweighted web link graphs. TextRank operates on undirected weighted graphs where edge weights reflect semantic or lexical co-occurrence similarity.

**Q2: Why does TextRank divide lexical overlap by $\log(|S_i|) + \log(|S_j|)$?**
> **Answer:** Long sentences naturally share more words by chance. Logarithmic normalization avoids penalizing long sentences too harshly while preventing them from dominating the similarity graph.